In [2]:
pip install -U bitsandbytes>=0.46.1

In [3]:
# Celda 1 — Cargar AR
# Notebook 3 — Celda 1: Cargar AR y datos
import torch, numpy as np, json, os, yaml
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from safetensors.torch import load_file
drive.mount('/content/drive')

BASE           = '/content/drive/MyDrive/nla_pipeline'
CHECKPOINT_AR  = f'{BASE}/checkpoints/nla_ar'
DIR_ACT        = f'{BASE}/activaciones'
DIR_VERB       = f'{BASE}/verbalizaciones'
DIR_RES        = f'{BASE}/resultados'
os.makedirs(DIR_RES, exist_ok=True)

# Leer sidecar del AR
with open(f'{CHECKPOINT_AR}/nla_meta.yaml') as f:
    ar_meta = yaml.safe_load(f)

MSE_SCALE      = ar_meta['extraction']['mse_scale']  # √3584 ≈ 59.87
AR_TEMPLATE    = ar_meta['prompt_templates']['ar']
print(f'mse_scale:     {MSE_SCALE:.4f}')
print(f'AR template:   {AR_TEMPLATE}')

# Cargar activaciones originales y verbalizaciones
activaciones = np.load(f'{DIR_ACT}/activaciones_L20.npy')   # [T, 3584]
with open(f'{DIR_VERB}/verbalizaciones.json', encoding='utf-8') as f:
    verbalizaciones = json.load(f)

print(f'\nActivaciones: {activaciones.shape}')
print(f'Verbalizaciones a procesar: {len(verbalizaciones)}')

# Detectar VRAM y cargar AR
vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
MODO_8BIT = vram_gb < 20
print(f'\nModo: {"8-bit" if MODO_8BIT else "bfloat16"}')

tok_ar = AutoTokenizer.from_pretrained(CHECKPOINT_AR, trust_remote_code=True)
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

if MODO_8BIT:
    ar_backbone = AutoModelForCausalLM.from_pretrained(
        CHECKPOINT_AR, quantization_config=quantization_config, device_map='auto',
        trust_remote_code=True,
    )
else:
    ar_backbone = AutoModelForCausalLM.from_pretrained(
        CHECKPOINT_AR, torch_dtype=torch.bfloat16,
        device_map='cuda:0', trust_remote_code=True,
    )

# Quitar la LayerNorm final (el AR ve la salida RAW del bloque, sin normalizar)
inner = ar_backbone.model
for attr in ('norm', 'final_layernorm', 'ln_f'):
    if hasattr(inner, attr):
        setattr(inner, attr, torch.nn.Identity())
        print(f'✓ LayerNorm final ({attr}) → Identity')
        break

# Cargar la cabeza de valor (value_head) — reconstruye el vector
d = ar_backbone.config.hidden_size   # 3584
value_head = torch.nn.Linear(d, d, bias=False, dtype=torch.float32)
vh_weights = load_file(f'{CHECKPOINT_AR}/value_head.safetensors')
value_head.load_state_dict(vh_weights)
device = next(ar_backbone.parameters()).device
value_head = value_head.to(device).eval()

ar_backbone.eval()
print(f'✓ AR listo. VRAM usada: {torch.cuda.memory_allocated()/1e9:.1f} GB')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
mse_scale:     59.8665
AR template:   Summary of the following text: <text>{explanation}</text> <summary>

Activaciones: (48, 3584)
Verbalizaciones a procesar: 38

Modo: 8-bit


Loading weights:   0%|          | 0/253 [00:00<?, ?it/s]

[transformers] Qwen2ForCausalLM LOAD REPORT from: /content/drive/MyDrive/nla_pipeline/checkpoints/nla_ar
Key               | Status  | 
------------------+---------+-
lm_head.weight    | MISSING | 
model.norm.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ LayerNorm final (norm) → Identity
✓ AR listo. VRAM usada: 7.1 GB


In [4]:
# Notebook 3 — Celda 2: Funciones de reconstrucción y score

@torch.inference_mode()
def reconstruir(descripcion):
    '''
    Texto → vector reconstruido.
    El AR usa el prompt: 'Summary of: <text>{desc}</text> <summary>'
    Extrae en el ÚLTIMO token → value_head → vector [3584].
    '''
    prompt = AR_TEMPLATE.format(explanation=descripcion)
    # add_special_tokens=True: Qwen tiene bos_token=None (no-op),
    # pero es importante para otros modelos — dejarlo siempre True
    ids = tok_ar(
        prompt,
        return_tensors='pt',
        add_special_tokens=True,
    )['input_ids'].to(device)

    # Forward pass — solo las primeras 21 capas (config del AR ya truncado)
    h = ar_backbone.model(ids, use_cache=False).last_hidden_state
    ultimo_token = h[0, -1]   # [3584] — extraer último token
    pred = value_head(ultimo_token.float()).cpu()   # [3584]
    return pred


def score(descripcion, v_raw_np):
    '''
    Calcula (MSE normalizado, cosine similarity) entre
    v_reconstruido y v_original.

    MSE = 2(1 - cos)  bajo normalización a mse_scale (√d_model).
    Rango: MSE ∈ [0, 4],  cos ∈ [-1, 1].
    '''
    pred = reconstruir(descripcion)                           # [3584]
    gold = torch.as_tensor(v_raw_np, dtype=torch.float32)    # [3584]

    # Normalizar ambos vectores a mse_scale = √3584 ≈ 59.87
    pred_n = pred / pred.norm().clamp_min(1e-12) * MSE_SCALE
    gold_n = gold / gold.norm().clamp_min(1e-12) * MSE_SCALE

    mse = ((pred_n - gold_n) ** 2).mean().item()
    cos = torch.nn.functional.cosine_similarity(
        pred.unsqueeze(0), gold.unsqueeze(0)
    ).item()
    return mse, cos

print('✓ Funciones reconstruir() y score() listas')



✓ Funciones reconstruir() y score() listas


In [5]:
# Notebook 3 — Celda 3: Evaluación y resultados finales
import csv

resultados_finales = []
print(f'Evaluando {len(verbalizaciones)} tokens...\n')

for item in verbalizaciones:
    pos         = item['posicion']
    descripcion = item['descripcion']
    v_raw       = activaciones[pos]   # [3584]

    mse, cos = score(descripcion, v_raw)

    # Interpretación automática
    if cos >= 0.9:    interp = '✓✓ EXCELENTE'
    elif cos >= 0.75: interp = '✓  BUENO'
    elif cos >= 0.5:  interp = '~  MEDIOCRE'
    else:             interp = '✗  POBRE'

    resultados_finales.append({
        'posicion':    pos,
        'norma_l2':    item['norma_l2'],
        'descripcion': descripcion,
        'cos_sim':     round(cos, 4),
        'mse':         round(mse, 4),
        'fidelidad':   interp,
    })
    print(f'pos={pos:2d} | cos={cos:.3f} | {interp}')
    print(f'        {descripcion[:70]}...')
    print()

# Guardar como JSON
ruta_json = f'{DIR_RES}/resultados_finales.json'
with open(ruta_json, 'w', encoding='utf-8') as f:
    json.dump(resultados_finales, f, ensure_ascii=False, indent=2)

# Guardar como CSV (más fácil de abrir en Excel/Sheets)
ruta_csv = f'{DIR_RES}/resultados_finales.csv'
with open(ruta_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=resultados_finales[0].keys())
    writer.writeheader()
    writer.writerows(resultados_finales)

# Resumen estadístico
cos_vals = [r['cos_sim'] for r in resultados_finales]
print('=' * 55)
print('RESUMEN FINAL')
print('=' * 55)
print(f'Tokens analizados:       {len(resultados_finales)}')
print(f'Cosine similarity media: {sum(cos_vals)/len(cos_vals):.4f}')
print(f'Cosine similarity máx:   {max(cos_vals):.4f}')
print(f'Cosine similarity mín:   {min(cos_vals):.4f}')
print(f'Resultados guardados en: {DIR_RES}')
print('\n🎉 Pipeline NLA completado con éxito!')



Evaluando 38 tokens...



/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


pos=10 | cos=0.878 | ✓  BUENO
        Structured AI-generated article with "Chinese Character" format and "a...

pos=11 | cos=0.821 | ✓  BUENO
        Chinese language model format with structured "definition" pattern est...

pos=12 | cos=0.773 | ✓  BUENO
        AI system prompt format with "I am an AI assistant created by Alibaba ...

pos=13 | cos=0.806 | ✓  BUENO
        Formal AI system introduction with "ChatGPT" and "Human-Oriented" fram...

pos=14 | cos=0.799 | ✓  BUENO
        Structured AI product description format with "assistant" identity and...

pos=15 | cos=0.771 | ✓  BUENO
        AI prompt structure with "Name: Qwen is a large language model" sugges...

pos=16 | cos=0.806 | ✓  BUENO
        Structured AI model prompt format with "Assistant: " label following a...

pos=17 | cos=0.787 | ✓  BUENO
        Formal AI platform UI context with a greeting ("I am Alpha GPT, a larg...

pos=18 | cos=0.726 | ~  MEDIOCRE
        Chinese language model format with "Answering with empa